# Train the "Truffles" wake word (openWakeWord)

Voice Control Prototype, phase 7 (Jamie Briggs, project 6074). Produces **`truffles.onnx`** for `models/` in the prototype.

**How to run**
1. *Runtime → Change runtime type → T4 GPU → Save.*
2. Leave `QUICK_TEST = True` in the first cell. *Runtime → Run all.* (~15 min: a rehearsal on tiny data, to catch problems cheaply.)
3. If it ends with **"Quick test passed"**, set `QUICK_TEST = False`, then *Runtime → Run all* again (~1.5–2.5 h). Keep the tab open.
4. At the end your browser downloads `truffles.onnx` and `training_report.json`.

If a cell stops with a red error, copy the error text and send it to Claude.

**What happens**: Piper (text-to-speech) says "truffles" thousands of times in hundreds of voices, plus sound-alikes ("trouble", "ruffles"…) as negatives; the clips are mixed with room echo and background audio; a small classifier is trained on top of openWakeWord's frozen feature model against ~2,000 h of other audio, and the version with the best balance of detections to false wake-ups is kept. Nothing about the prototype or your voice is uploaded.

*This is a repaired version of openWakeWord's `automatic_model_training.ipynb` (8 Oct 2026). Fixes: Python 3.12+ (`piper-phonemize-fix`), piper-sample-generator pinned to v2.0.0, PyTorch/torchaudio/torchvision pinned to 2.8 / 0.23 (`torchaudio.load/info` still exist; old `torch.load` behaviour allowed for the trusted Piper voice file), dead AudioSet/FMA download links replaced, no TensorFlow (only needed for the unused .tflite export), separate folders for the quick test and the real run.*

In [ ]:
# ---- Settings: the only cell you might edit -------------------------------
QUICK_TEST = True        # True = ~15 min rehearsal on tiny data. Then set False and "Run all" again.

PHRASE = "truffles"      # exactly what the robot answers to
MODEL_NAME = "truffles"  # the file will be truffles.onnx

# Sound-alikes the model must learn to IGNORE (most from the CMU pronouncing dictionary).
NEAR_MISSES = ["truffle", "trouble", "troubles", "ruffles", "shuffles", "scuffles",
               "trifles", "rifles", "struggles", "truffle oil", "true falls", "tough calls"]

# Size of the real run. Bigger = better model but slower; these fit a free T4 in ~1.5-2.5 h.
N_SAMPLES = 6000         # "truffles" clips (and as many near-miss clips) for training
N_SAMPLES_VAL = 1500     # held-out clips for choosing the best model
STEPS = 30000            # training steps

In [ ]:
# Check the machine, and define a helper that stops "Run all" at the first failure.
import os, subprocess, sys
print("Python", sys.version.split()[0])
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
assert "GPU" in gpu.stdout, "No GPU: Runtime > Change runtime type > T4 GPU, then Runtime > Run all"

# PyTorch 2.6+ refuses to load the Piper voice file (a whole pickled model) unless told
# it is trusted. It is: it comes from the Piper project's own GitHub release.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.chdir("/content")

def run(cmd):
    """Run a shell command; stop everything if it fails."""
    print("$", cmd)
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError("That step failed. Copy the error text above and send it to Claude.")

In [ ]:
%%bash
# Install the training tools (~3-5 min). Versions pinned and checked on 8 Oct 2026.
set -euo pipefail
cd /content
# PyTorch 2.8: the last version whose torchaudio still has load()/info(), used by openWakeWord's training code.
# torchvision must match it too: Colab's own torchvision is built for a newer PyTorch and crashes
# ("operator torchvision::nms does not exist") when speechbrain imports it.
pip install -q "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0"
# piper-phonemize-fix = the same piper_phonemize module, built for Python 3.12+ (the original stops at 3.11).
pip install -q piper-phonemize-fix==1.2.2 webrtcvad-wheels mutagen==1.47.0 torchinfo==1.8.0 \
    torchmetrics==1.2.0 speechbrain==0.5.16 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
    pronouncing==0.2.0 acoustics==0.2.6 onnx onnxruntime soundfile pyarrow huggingface_hub pyyaml
# openWakeWord source, pinned commit, without its run-time extras (tflite, speexdsp).
[ -d openwakeword ] || git clone -q https://github.com/dscripka/openWakeWord openwakeword
git -C openwakeword checkout -q 368c03716d1e92591906a84949bc477f3a834455
# Bug fix: train.py's flags default to the TEXT "False", which Python treats as true, so it always
# tried the TFLite conversion (needs TensorFlow) after saving the ONNX model. Make them real False.
sed -i 's/default="False"/default=False/' openwakeword/openwakeword/train.py
pip install -q --no-deps -e ./openwakeword
# Piper sample generator v2.0.0: the version train.py expects (newer ones moved generate_samples.py).
[ -d piper-sample-generator ] || git clone -q --depth 1 --branch v2.0.0 https://github.com/rhasspy/piper-sample-generator
[ -f piper-sample-generator/models/en_US-libritts_r-medium.pt ] || wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
    https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
# openWakeWord's frozen "ear": mel spectrogram + embedding models.
M=openwakeword/openwakeword/resources/models; mkdir -p $M
for f in melspectrogram.onnx embedding_model.onnx; do
  [ -f $M/$f ] || wget -q -O $M/$f https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/$f
done
python -c "import torch, torchaudio; print('torch', torch.__version__, '| GPU', torch.cuda.is_available(), '| torchaudio', torchaudio.__version__)"
python -c "import sys; sys.path.insert(0, 'piper-sample-generator'); import generate_samples, openwakeword.data; print('training imports OK')"

In [ ]:
%%writefile prepare_data.py
"""Download everything wake-word training needs (runs on Google Colab, not the Mac).

Called by truffles_wake_word.ipynb:   python prepare_data.py [--quick]

  mit_rirs/          room echoes (impulse responses), to make clips sound like real rooms
  background_16k/    everyday background audio (speech, music, noise) from AudioSet
  acav_features.npy  ~2,000 h of "not the wake word" audio, already passed through
                     openWakeWord's frozen "ear" (17 GB; --quick takes only the start)
  validation_set_features.npy  ~11 h of other audio, used to count false wake-ups

Written 8 Oct 2026 because the official notebook's downloads had broken: its
AudioSet .tar link is gone (now Parquet files) and its music source is a
`datasets` loading script that new versions no longer run. This uses
huggingface_hub directly instead of `datasets`.
"""

import argparse
import io
import os
import shutil
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import requests
import soundfile as sf
from huggingface_hub import hf_hub_download, snapshot_download
from scipy.signal import resample_poly

FEATURES = "davidscripka/openwakeword_features"


def room_echoes():
    out = Path("mit_rirs")
    if out.exists() and any(out.glob("*.wav")):
        return
    folder = snapshot_download("davidscripka/MIT_environmental_impulse_responses", repo_type="dataset",
                               allow_patterns=["16khz/*"], local_dir="mit_rirs_download")
    out.mkdir(exist_ok=True)
    for f in Path(folder, "16khz").glob("*.wav"):
        shutil.copy(f, out / f.name)
    print("room echoes:", len(list(out.glob("*.wav"))))


def to_16k_mono(audio, rate):
    """Training expects 16 kHz mono, like the prototype."""
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if rate != 16000:
        g = np.gcd(int(rate), 16000)
        audio = resample_poly(audio, 16000 // g, int(rate) // g)
    return np.clip(audio, -1, 1)


def background(n_files, max_clips):
    """AudioSet's balanced set: 10 s YouTube clips of every kind of sound, music included."""
    out = Path("background_16k")
    out.mkdir(exist_ok=True)
    count = len(list(out.glob("*.wav")))
    for i in range(n_files):
        if count >= max_clips:
            break
        path = hf_hub_download("agkphysics/AudioSet", f"data/bal_train/{i:02d}.parquet",
                               repo_type="dataset", local_dir="audioset")
        for batch in pq.ParquetFile(path).iter_batches(batch_size=32, columns=["video_id", "audio"]):
            for vid, audio in zip(batch.column("video_id").to_pylist(), batch.column("audio").to_pylist()):
                try:
                    data, rate = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
                except Exception:
                    continue                      # the odd undecodable clip is just skipped
                sf.write(out / f"{vid}.wav", to_16k_mono(data, rate), 16000, subtype="PCM_16")
                count += 1
                if count >= max_clips:
                    break
            if count >= max_clips:
                break
        os.remove(path)                           # keep the Colab disk tidy
    if count == 0:
        raise SystemExit("No background clips could be decoded: send this error to Claude.")
    print("background clips:", count)


def npy_head(url, out, rows):
    """Download only the first `rows` rows of a huge .npy file (for the quick test).
    A .npy file is a small header followed by the rows, so two HTTP range requests do it."""
    if Path(out).exists():
        return
    f = io.BytesIO(requests.get(url, headers={"Range": "bytes=0-4095"}, timeout=60).content)
    major, _ = np.lib.format.read_magic(f)
    read_header = np.lib.format.read_array_header_1_0 if major == 1 else np.lib.format.read_array_header_2_0
    shape, fortran, dtype = read_header(f)
    start, row_bytes = f.tell(), int(np.prod(shape[1:])) * dtype.itemsize
    rows = min(rows, shape[0])
    data = requests.get(url, headers={"Range": f"bytes={start}-{start + rows * row_bytes - 1}"}, timeout=600).content
    np.save(out, np.frombuffer(data, dtype).reshape((rows,) + tuple(shape[1:])))
    print(f"{out}: first {rows} of {shape[0]} rows")


def features(quick):
    if not Path("validation_set_features.npy").exists():
        hf_hub_download(FEATURES, "validation_set_features.npy", repo_type="dataset", local_dir=".")
    if quick:
        url = f"https://huggingface.co/datasets/{FEATURES}/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
        npy_head(url, "acav_features.npy", 20000)
    elif not Path("acav_features.npy").exists() or Path("acav_features.npy").stat().st_size < 1e10:
        path = hf_hub_download(FEATURES, "openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
                               repo_type="dataset", local_dir=".")
        os.replace(path, "acav_features.npy")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--quick", action="store_true", help="tiny data for the rehearsal run")
    quick = ap.parse_args().quick
    room_echoes()
    background(n_files=1 if quick else 2, max_clips=100 if quick else 3000)
    features(quick)
    print("data ready")

In [ ]:
# Download the data (quick: ~1 GB, a few minutes; real run: ~19 GB, ~15-30 min).
run("python prepare_data.py" + (" --quick" if QUICK_TEST else ""))

In [ ]:
# Write the training settings (openWakeWord's defaults, with ours on top).
import yaml
OUT_DIR = "quick_test_model" if QUICK_TEST else "my_custom_model"   # separate, so the real run never reuses rehearsal data
config = yaml.safe_load(open("openwakeword/examples/custom_model.yml"))
config.update({
    "model_name": MODEL_NAME,
    "target_phrase": [PHRASE],
    "custom_negative_phrases": NEAR_MISSES,
    "n_samples": 60 if QUICK_TEST else N_SAMPLES,
    "n_samples_val": 30 if QUICK_TEST else N_SAMPLES_VAL,
    "steps": 200 if QUICK_TEST else STEPS,
    "tts_batch_size": 50,
    "piper_sample_generator_path": "./piper-sample-generator",
    "output_dir": OUT_DIR,
    "rir_paths": ["./mit_rirs"],
    "background_paths": ["./background_16k"],
    "background_paths_duplication_rate": [1],
    "false_positive_validation_data_path": "./validation_set_features.npy",
    "feature_data_files": {"ACAV100M_sample": "./acav_features.npy"},
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
})
yaml.safe_dump(config, open("truffles.yaml", "w"), sort_keys=False)
print(yaml.safe_dump(config, sort_keys=False))

In [ ]:
# Step 1: make the synthetic clips with Piper (real run: ~30-60 min on a T4).
# If Colab disconnects, just "Run all" again: it carries on from the clips it already has.
run("python openwakeword/openwakeword/train.py --training_config truffles.yaml --generate_clips")

In [ ]:
# Step 2: add room echo and background noise, then turn every clip into features (~10-30 min).
run("python openwakeword/openwakeword/train.py --training_config truffles.yaml --augment_clips")

In [ ]:
# Step 3: train the model (~20-60 min). It keeps the version with the best balance of
# detections to false wake-ups, and saves it as ONNX (no --convert_to_tflite: we don't need TFLite).
run("python openwakeword/openwakeword/train.py --training_config truffles.yaml --train_model")

In [ ]:
%%writefile check_model.py
"""First sanity check of a freshly trained wake-word model (runs on Colab).

    python check_model.py <output_dir> <model_name>

Plays held-out SYNTHETIC test clips (made during training but never trained on)
through the model the same way the prototype does (80 ms chunks), and writes
training_report.json. These numbers are an optimistic first look: the real
test is Jamie's voice, in the prototype's benchmark.
"""

import glob
import json
import sys

# This script sits in /content next to the openWakeWord *repository* folder, which is
# also called "openwakeword". Python searches the script's own folder first, so it would
# import that folder instead of the installed package. Remove the script's folder from
# the search path (found on the first Colab run: "has no attribute get_pretrained_model_paths").
sys.path.pop(0)

import numpy as np
import soundfile as sf
from openwakeword.model import Model

out_dir, name = sys.argv[1], sys.argv[2]
model = Model(wakeword_models=[f"{out_dir}/{name}.onnx"], inference_framework="onnx")


def peak_score(path):
    """Highest score the model gives anywhere in the clip (with 1 s of silence either side)."""
    audio, _ = sf.read(path, dtype="int16")
    pad = np.zeros(16000, np.int16)
    audio = np.concatenate([pad, audio, pad])
    model.reset()
    return max(list(model.predict(audio[i:i + 1280]).values())[0]
               for i in range(0, len(audio) - 1279, 1280))


positives = [peak_score(p) for p in sorted(glob.glob(f"{out_dir}/{name}/positive_test/*.wav"))[:300]]
near_misses = [peak_score(p) for p in sorted(glob.glob(f"{out_dir}/{name}/negative_test/*.wav"))[:300]]

report = {"model": name, "synthetic_test_clips": True,
          "positive_clips": len(positives), "near_miss_clips": len(near_misses)}
for threshold in (0.3, 0.5, 0.7):
    report[f"detected_at_{threshold}"] = round(float(np.mean([s >= threshold for s in positives])), 3)
    report[f"near_miss_fired_at_{threshold}"] = round(float(np.mean([s >= threshold for s in near_misses])), 3)
json.dump(report, open(f"{out_dir}/training_report.json", "w"), indent=2)
print(json.dumps(report, indent=2))

In [ ]:
# Step 4: first check on held-out synthetic clips, then download the results.
run(f"python check_model.py {OUT_DIR} {MODEL_NAME}")
if QUICK_TEST:
    print("\n*** Quick test passed. *** (The model above is a toy: ignore its numbers.)")
    print("Now set QUICK_TEST = False in the first cell, then Runtime > Run all.")
else:
    import shutil
    from google.colab import files
    shutil.copy(f"truffles.yaml", f"{OUT_DIR}/truffles_training_config.yaml")
    for f in (f"{MODEL_NAME}.onnx", "training_report.json", "truffles_training_config.yaml"):
        files.download(f"{OUT_DIR}/{f}")
    print("Done: put truffles.onnx in the prototype's models/ folder and restart the app.")